# EEG-Conformer Seizure Prediction
Cross-patient benchmarking on CHB-MIT · 20 seeds · mean ± std

Architecture: Song et al. (2023), IEEE TNSRE — CNN + Transformer self-attention.

In [ ]:
from pathlib import Path
import sys

def find_repo_root(start):
    for path in [start, *start.parents]:
        if (path / 'src').exists() and (path / 'README.md').exists():
            return path
    return start

ROOT = find_repo_root(Path.cwd())
sys.path.insert(0, str(ROOT / 'src'))


In [ ]:
import os
import json
import math
import numpy as np
import torch
import torch.nn as nn
from torch.optim import AdamW
from sklearn.metrics import roc_auc_score
from data_utils_leaky import get_leaky_dataloaders_for_seed as get_dataloaders_for_seed
from data_utils import SEEDS, DATA_DIR
from eval_utils import find_youden_threshold, full_evaluate

DEVICE       = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
N_CHANNELS   = 18
WIN          = 20 * 256   # 5120
BATCH_SIZE   = 64         # Conformer is heavier — reduce batch size to fit 8GB VRAM
MAX_EPOCHS   = 100
PATIENCE     = 20
LR           = 1e-4
WEIGHT_DECAY = 1e-4
OUT_DIR      = r'D:\seizure_results\eeg_conformer_leaky'

print(f'Device : {DEVICE}')
print(f'Seeds  : {len(SEEDS)}')
os.makedirs(OUT_DIR, exist_ok=True)

In [ ]:
class PatchEmbedding(nn.Module):
    """
    Converts (B, C, T) EEG into (B, S, D) token sequence.
    Step 1 — temporal conv (1 x kern_t): learn filter bank across time.
    Step 2 — spatial conv (n_chans x 1): project across electrodes.
    Step 3 — BN + ELU + AvgPool: downsample time into S tokens.
    """
    def __init__(self,
                 n_channels  = N_CHANNELS,
                 emb_size    = 40,
                 kern_t      = 15,
                 pool_size   = 8,
                 pool_stride = 8,
                 dropout     = 0.5):
        super().__init__()
        # Input shape: (B, C, T) → unsqueeze → (B, 1, C, T)
        self.temporal_conv = nn.Sequential(
            nn.Conv2d(1, emb_size, kernel_size=(1, kern_t),
                      padding=(0, kern_t // 2), bias=False),
            nn.BatchNorm2d(emb_size),
        )
        # Spatial conv: collapses channel dimension
        self.spatial_conv = nn.Sequential(
            nn.Conv2d(emb_size, emb_size, kernel_size=(n_channels, 1), bias=False),
            nn.BatchNorm2d(emb_size),
            nn.ELU(),
            nn.Dropout(dropout),
        )
        # AvgPool along time: (B, D, 1, T) → (B, D, 1, S)
        self.pool = nn.AvgPool2d(kernel_size=(1, pool_size),
                                 stride=(1, pool_stride))

    def forward(self, x):
        # x: (B, C, T)
        x = x.unsqueeze(1)                  # (B, 1, C, T)
        x = self.temporal_conv(x)           # (B, D, C, T)
        x = self.spatial_conv(x)            # (B, D, 1, T)
        x = self.pool(x)                    # (B, D, 1, S)
        x = x.squeeze(2)                    # (B, D, S)
        x = x.permute(0, 2, 1)             # (B, S, D)  
        return x


class MultiHeadAttention(nn.Module):
    """Standard scaled dot-product multi-head attention."""
    def __init__(self, emb_size, num_heads, dropout=0.5):
        super().__init__()
        self.attn    = nn.MultiheadAttention(emb_size, num_heads,
                                              dropout=dropout, batch_first=True)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out, _ = self.attn(x, x, x)
        return self.dropout(out)


class FeedForward(nn.Module):
    """Position-wise feed-forward block (expansion factor = 4)."""
    def __init__(self, emb_size, expansion=4, dropout=0.5):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(emb_size, emb_size * expansion),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(emb_size * expansion, emb_size),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)


class TransformerEncoderBlock(nn.Module):
    """Single Transformer encoder block with pre-norm (LayerNorm before sublayer)."""
    def __init__(self, emb_size, num_heads, dropout=0.5):
        super().__init__()
        self.norm1 = nn.LayerNorm(emb_size)
        self.attn  = MultiHeadAttention(emb_size, num_heads, dropout)
        self.norm2 = nn.LayerNorm(emb_size)
        self.ff    = FeedForward(emb_size, dropout=dropout)

    def forward(self, x):
        x = x + self.attn(self.norm1(x))
        x = x + self.ff(self.norm2(x))
        return x


class TransformerEncoder(nn.Module):
    def __init__(self, depth, emb_size, num_heads, dropout=0.5):
        super().__init__()
        self.layers = nn.ModuleList([
            TransformerEncoderBlock(emb_size, num_heads, dropout)
            for _ in range(depth)
        ])

    def forward(self, x):
        for layer in self.layers:
            x = layer(x)
        return x


class ClassificationHead(nn.Module):
    """Flatten sequence + MLP classifier (original: FC → ELU → Dropout → FC)."""
    def __init__(self, emb_size, n_tokens, n_classes=2, dropout=0.5):
        super().__init__()
        flat_dim = emb_size * n_tokens
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(flat_dim, 256),
            nn.ELU(),
            nn.Dropout(dropout),
            nn.Linear(256, 32),
            nn.ELU(),
            nn.Dropout(dropout),
            nn.Linear(32, n_classes),
        )

    def forward(self, x):
        return self.net(x)


class EEGConformer(nn.Module):
    """
    EEG-Conformer — Song et al. (2023), IEEE TNSRE.
    Original hyperparameters: emb_size=40, depth=6, heads=10, kern_t=15.
    Pool stride adapted for 20s@256Hz (5120 samples) to yield ~40 tokens.

    Input  : (B, 18, 5120)
    Output : (B, 2)
    """
    def __init__(self,
                 n_channels  = N_CHANNELS,
                 n_times     = WIN,
                 emb_size    = 40,
                 depth       = 6,
                 num_heads   = 10,
                 kern_t      = 15,
                 pool_size   = 75,
                 pool_stride = 75,
                 dropout     = 0.5,
                 n_classes   = 2):
        super().__init__()

        self.patch_embed = PatchEmbedding(
            n_channels  = n_channels,
            emb_size    = emb_size,
            kern_t      = kern_t,
            pool_size   = pool_size,
            pool_stride = pool_stride,
            dropout     = dropout,
        )

        # Compute actual token count dynamically
        with torch.no_grad():
            dummy   = torch.zeros(1, n_channels, n_times)
            tokens  = self.patch_embed(dummy)   # (1, S, D)
            n_tokens = tokens.shape[1]

        self.transformer = TransformerEncoder(depth, emb_size, num_heads, dropout)
        self.classifier  = ClassificationHead(emb_size, n_tokens, n_classes, dropout)

    def forward(self, x):
        x = self.patch_embed(x)     # (B, S, D)
        x = self.transformer(x)     # (B, S, D)
        return self.classifier(x)   # (B, n_classes)


# Shape and parameter check
dummy = torch.zeros(4, N_CHANNELS, WIN)
model_test = EEGConformer()
out = model_test(dummy)
print('Output shape  :', out.shape)   # expect (4, 2)
n_params = sum(p.numel() for p in model_test.parameters() if p.requires_grad)
print(f'Parameters    : {n_params:,}')
with torch.no_grad():
    tokens = model_test.patch_embed(dummy)
print(f'Token sequence: {tokens.shape[1]} tokens × {tokens.shape[2]} dim')

In [ ]:
def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0.0
    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(model(x), y)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        total_loss += loss.item() * len(y)
    return total_loss / len(loader.dataset)


@torch.no_grad()
def collect_probs(model, loader):
    model.eval()
    all_probs, all_labels = [], []
    for x, y in loader:
        logits = model(x.to(DEVICE))
        probs = torch.softmax(logits, dim=1)[:, 1].cpu().numpy()
        all_probs.append(probs)
        all_labels.append(y.numpy())
    return np.concatenate(all_probs), np.concatenate(all_labels)


print('Helpers defined.')

In [ ]:
def run_seed(seed):
    print(f"\n{'='*60}")
    print(f"  Seed {seed}")
    print(f"{'='*60}")

    (train_loader, val_loader, test_loader,
     n_train, n_val, n_test,
     n_pre, n_inter) = get_dataloaders_for_seed(seed, DATA_DIR, BATCH_SIZE)

    _bx, _by = next(iter(train_loader))
    _n1 = int(_by.sum())
    print(f"  Batch check : {_n1}/{len(_by)} pre-ictal ({100*_n1/len(_by):.0f}%)")
    del _bx, _by

    model     = EEGConformer().to(DEVICE)
    optimizer = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='max', factor=0.5, patience=5, min_lr=1e-7)
    criterion = nn.CrossEntropyLoss()

    best_val_auc  = 0.0
    best_state    = {k: v.cpu().clone() for k, v in model.state_dict().items()}
    patience_left = PATIENCE

    for epoch in range(1, MAX_EPOCHS + 1):
        train_loss = train_one_epoch(model, train_loader, optimizer, criterion)
        val_probs, val_labels = collect_probs(model, val_loader)
        val_auc = roc_auc_score(val_labels, val_probs)
        scheduler.step(val_auc)
        current_lr = optimizer.param_groups[0]['lr']

        print(f"  Epoch {epoch:3d} | loss {train_loss:.4f} | "
              f"val AUC {val_auc:.4f} | lr {current_lr:.2e}")

        if val_auc > best_val_auc:
            best_val_auc  = val_auc
            best_state    = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            patience_left = PATIENCE
        else:
            patience_left -= 1
            if patience_left == 0:
                print(f"  Early stopping at epoch {epoch}.")
                break

    model.load_state_dict(best_state)
    model.to(DEVICE)

    val_probs, val_labels = collect_probs(model, val_loader)
    threshold = find_youden_threshold(val_labels, val_probs)
    print(f"  Youden threshold (val): {threshold:.4f}")

    test_probs, test_labels = collect_probs(model, test_loader)
    m = full_evaluate(test_labels, test_probs, threshold, stride_s=300)

    print(f"\n  [Seed {seed}] TEST  AUC {m['auc']:.4f} | "
          f"Sen {m['sensitivity']:.4f} | Spe {m['specificity']:.4f} | "
          f"Prec {m['precision']:.4f} | F1 {m['f1']:.4f} | "
          f"FAR {m['far']:.3f}/h | "
          f"EvtSen {m['event_sensitivity']:.3f} ({m['n_events']} events)")

    torch.save(best_state, os.path.join(OUT_DIR, f'seed{seed}_best.pt'))

    return {
        'seed':               seed,
        'test_auc':           m['auc'],
        'test_sen':           m['sensitivity'],
        'test_spe':           m['specificity'],
        'test_precision':     m['precision'],
        'test_f1':            m['f1'],
        'far':                m['far'],
        'event_sensitivity':  m['event_sensitivity'],
        'n_events':           m['n_events'],
        'threshold':          m['threshold'],
        'best_val_auc':       best_val_auc,
    }

print('run_seed defined.')

In [ ]:
all_results = []
for s in [42]:
    result = run_seed(s)
    all_results.append(result)

results_path = os.path.join(OUT_DIR, 'results.json')
with open(results_path, 'w') as _f:
    json.dump(
        [{k: (int(v) if k in ('seed', 'n_events') else float(v))
          for k, v in r.items()}
         for r in all_results],
        _f, indent=2,
    )
print(f'\nAll results saved to {results_path}')

In [ ]:
metrics = ['test_auc', 'test_sen', 'test_spe', 'test_precision',
           'test_f1', 'far', 'event_sensitivity']
labels  = ['AUC', 'Sensitivity (win)', 'Specificity', 'Precision',
           'F1', 'FAR (/h)', 'Sensitivity (event)']

print(f'EEG-Conformer  Cross-Patient Results (mean ± std, n={len(all_results)} seeds)')
print('-' * 55)
for m, l in zip(metrics, labels):
    vals = np.array([r[m] for r in all_results])
    print(f'  {l:<22s}: {vals.mean():.4f} ± {vals.std(ddof=1):.4f}')